# 👵 Nursing Home Fall Detection - YOLO11 Training (EXP-006: Hybrid Optimization)

Notebook ini digunakan untuk melatih model **YOLO11 Nano** dengan optimasi **EXP-006: The Hybrid** (AdamW + Cosine LR + Extended Patience 30 + Night/IR Augmentation + Class Loss Tuning).

### 📌 Petunjuk Penggunaan di Kaggle:
1. Nyalakan GPU di menu: **Settings -> Accelerator -> GPU T4 x1**.
2. Pastikan dataset sudah ditambahkan via menu: **+ Add Data** -> pilih dataset `fall-detection-dataset` (Path: `/kaggle/input/datasets/rajarahmanaziiz/fall-detection-dataset`).
3. Dataset langsung terbaca dari folder (tidak perlu diekstrak dari .zip).
4. Jalankan cell secara berurutan dari atas ke bawah.

### 1. Install Ultralytics & Verifikasi Akselerasi GPU

In [1]:
!pip install -q ultralytics

import torch
import yaml
from pathlib import Path
from ultralytics import YOLO

print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name     :', torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch Version : 2.10.0+cu128
CUDA Available  : True
Device Name     : Tesla T4


### 2. Deteksi Dataset Kaggle (Folder Langsung / Uncompressed)

In [2]:
# Path utama dataset Kaggle
primary_dataset_path = Path('/kaggle/input/datasets/rajarahmanaziiz/fall-detection-dataset')
fallback_candidates = [
    primary_dataset_path,
    Path('/kaggle/input/fall-detection-dataset'),
    Path('/kaggle/input/urfall-yolo-dataset'),
]

dataset_dir = None

# Cek path utama atau fallback folder
for candidate in fallback_candidates:
    if candidate.exists():
        if (candidate / 'images').exists():
            dataset_dir = candidate
            break
        sub_images = list(candidate.glob('**/images'))
        if sub_images:
            dataset_dir = sub_images[0].parent
            break

if not dataset_dir:
    kaggle_input = Path('/kaggle/input')
    found_images = list(kaggle_input.glob('**/images'))
    if found_images:
        dataset_dir = found_images[0].parent

if dataset_dir:
    print(f'✅ Dataset berhasil ditemukan di: {dataset_dir}')
    print(f'   Train images : {len(list((dataset_dir / "images" / "train").glob("*")))} file')
    print(f'   Val images   : {len(list((dataset_dir / "images" / "val").glob("*")))} file')
    print(f'   Test images  : {len(list((dataset_dir / "images" / "test").glob("*")))} file')
else:
    raise FileNotFoundError('❌ Folder dataset tidak ditemukan di /kaggle/input! Periksa input dataset Anda.')


✅ Dataset berhasil ditemukan di: /kaggle/input/datasets/rajarahmanaziiz/fall-detection-dataset
   Train images : 924 file
   Val images   : 254 file
   Test images  : 320 file


### 3. Generate data.yaml di Lingkungan Kaggle

In [3]:
work_dir = Path('/kaggle/working')
kaggle_yaml_path = work_dir / 'data.yaml'

data_yaml_content = {
    'path': str(dataset_dir.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': {
        0: 'normal',
        1: 'transitional',
        2: 'lying_on_ground'
    }
}

with open(kaggle_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f'✓ data.yaml berhasil dibuat di: {kaggle_yaml_path}')
print('-' * 35)
print(open(kaggle_yaml_path).read())
print('-' * 35)


✓ data.yaml berhasil dibuat di: /kaggle/working/data.yaml
-----------------------------------
names:
  0: normal
  1: transitional
  2: lying_on_ground
nc: 3
path: /kaggle/input/datasets/rajarahmanaziiz/fall-detection-dataset
test: images/test
train: images/train
val: images/val

-----------------------------------


### 4. Training EXP-006: The Hybrid Optimization

> 🔬 **HIPOTESIS EXP-006:**
> - **Model**: `yolo11n.pt` (terbukti tidak overfit pada 924 sampel).
> - **Optimizer**: `AdamW` + `cos_lr=True` (`lr0=0.001`) untuk kestabilan gradient dan menaikkan Precision.
> - **Patience 30 & Epochs 70**: Mempertahankan Recall rekor tinggi seperti EXP-005 (>= 90%).
> - **Augmentasi Realita Panti**: `hsv_v=0.6` (cahaya minim), `bgr=0.2` (mode infrared CCTV malam), `erasing=0.4` (occlusion selimut/kursi).
> - **Class Weight (`cls=1.0`)**: Mempertajam pemisahan antara postur `lying_on_ground` dan `transitional`.

In [4]:
print('🚀 Memulai Training EXP-006...')
model = YOLO('yolo11n.pt')

results = model.train(
    data=str(kaggle_yaml_path),
    epochs=70,
    patience=30,
    batch=16,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    cls=1.0,
    hsv_v=0.6,
    hsv_s=0.7,
    hsv_h=0.015,
    bgr=0.2,
    erasing=0.4,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
    project='/kaggle/working/models',
    name='exp006_hybrid_adamw',
    exist_ok=True,
    save=True
)

print('\n🎉 Training EXP-006 selesai!')
print(f'Hasil tersimpan di: {results.save_dir}')


🚀 Memulai Training EXP-006...
Ultralytics 8.4.171 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.2, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=1.0, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=70, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.6, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp006_hybrid_adamw, nbs=64, nms=No

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/70      2.53G     0.8461       1.94      1.087         23        640: 100% ━━━━━━━━━━━━ 58/58 8.0it/s 7.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 6.0it/s 1.3s
                   all        254        254      0.648      0.535      0.425       0.35

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/70      2.53G     0.8235       1.76      1.055         23        640: 100% ━━━━━━━━━━━━ 58/58 8.5it/s 6.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 6.0it/s 1.3s
                   all        254        254      0.824      0.354      0.418      0.328

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/70      2.53G     0.8004      1.676      1.063         32        640: 100% ━━━━━━━━━━━━ 58/58 8.4it/s 6.9s
                 Class     Images  Instances      Box(P          R

### 5. Evaluasi pada Test Set & Cetak Metrik Lengkap

In [5]:
print('🔍 Menjalankan evaluasi pada Test Set...')
metrics = model.val(data=str(kaggle_yaml_path), split='test')

print('\n' + '='*50)
print('📊 HASIL METRIK TEST SET EXP-006:')
print('='*50)
print(f'• Test mAP50    : {metrics.box.map50 * 100:.2f}%')
print(f'• Test mAP50-95 : {metrics.box.map * 100:.2f}%')
print(f'• Test Precision: {metrics.box.mp * 100:.2f}%')
print(f'• Test Recall   : {metrics.box.mr * 100:.2f}%')
print('='*50)
print('\n📁 Model terbaik kalian tersimpan di:')
print(f'{results.save_dir}/weights/best.pt')
print('Silakan unduh best.pt dan grafik evaluasi dari tab Output Kaggle!')


🔍 Menjalankan evaluasi pada Test Set...
Ultralytics 8.4.171 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.2±0.5 ms, read: 16.6±6.9 MB/s, size: 61.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/rajarahmanaziiz/fall-detection-dataset/labels/test... 320 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 320/320 392.3it/s 0.8s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/rajarahmanaziiz/fall-detection-dataset/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 5.9it/s 3.4s
                   all        320        320      0.686      0.763      0.702      0.617
                normal         27    